# 第21章 GARCH 模型——波动率的聚类现象

> **动机先行**: 第20章的结论看似令人沮丧——沪深300日收益率是白噪声, ARIMA 什么也预测不了。但这是否意味着金融数据中"完全没有结构"？绝非如此。回到第19章的一个关键发现: 收益率**本身**的 ACF 不显著, 但收益率**绝对值**(或平方)的 ACF 高度显著且持续数十期——这就是**波动率聚集 (Volatility Clustering)**: 大波动之后往往跟着大波动, 小波动之后往往跟着小波动。本章的 GARCH 模型正是为了建模这种"波动率本身的 ARMA 结构"而设计的——它不预测涨跌方向, 而是预测**涨跌幅度的大小**。
>
> **量化实战定位**: GARCH 是量化风控的基石。VaR(在险价值)的计算、期权的隐含波动率曲面校准、组合的压力测试——这些日常风控操作都依赖 GARCH 或其变体对波动率的实时估计。Engle 因 ARCH 模型获得 2003 年诺贝尔经济学奖, 获奖词的核心就是:"他教会了我们如何度量风险的时变性。"

---

## 21.1 动机: 收益率不能预测, 但波动率可以

第19-20章聚焦于一个问题的两个层面——今天和昨天"方向"上的关系。结论是: 日频收益率在方向上几乎是白噪声。但金融从业者凭直觉就知道"市场有平静期和风暴期的交替"——这不需要数学, 看 K 线图就能感受到。

这种交替的统计证据来自**收益率平方的自相关**。第19章 Ljung-Box 检验的结果已经揭示了这一点:

| 序列 | Ljung-Box $Q(20)$ | p 值 | 结论 |
|------|-------------------|------|------|
| $r_t$ (收益率) | 26.40 | 0.15 | 白噪声——无自相关 |
| $\|r_t\|$ (绝对值) | 242.05 | 0.00 | 强烈自相关——波动率有记忆 |

**波动率聚集的直观解释**: 一个重大利空消息(比如突发政策变化)会引发市场恐慌, 接下来几天投资者持续消化信息、调整仓位, 市场波动加大。等事件影响消退, 市场回归平静。从统计上看, 这就表现为波动率的**自回归**行为——今天的波动率 = 昨天的波动率 × 某个衰减系数 + 新的冲击。

这正是 GARCH 模型的直觉: **方差本身遵循一个 ARMA 式的递归方程。**

---

## 21.2 从同方差到异方差: 为什么 OLS 的假设在金融中不成立

### 21.2.1 条件异方差的含义

第12章回归分析的经典假设之一是**同方差性 (Homoskedasticity)**: 误差项的方差恒定, $Var(\varepsilon_t) = \sigma^2$(一个常数)。但金融数据的方差显然不是常数:

- 2008 年金融危机期间, 美股日波动率是平时的 3-5 倍
- A 股 2015 年股灾期间, 千股跌停的日子波动率飙升
- 即使是"正常"时期, 低波动和高波动时段也交替出现

**条件异方差 (Conditional Heteroskedasticity)** 正是对此的数学描述:

$$Var(\varepsilon_t \mid \mathcal{F}_{t-1}) \neq \text{常数}$$

其中 $\mathcal{F}_{t-1}$ 表示"截至 $t-1$ 期所有已知信息"。简单说: **给定今天之前的信息, 明天的波动率是可以预测的**, 它依赖于过去的波动率和过去的冲击。

### 21.2.2 用数据可视化波动率聚集

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from statsmodels.graphics.tsaplots import plot_acf
plt.rcParams['font.sans-serif'] = ['WenQuanYi Micro Hei']
plt.rcParams['axes.unicode_minus'] = False

csv_path = 'data/index_data_7_20210601_20260531.csv'
df = pd.read_csv(csv_path, parse_dates=['time'])
hs300_ret = np.log(df[df['thscode']=='000300.SH'].set_index('time')['close']).diff().dropna()
abs_ret = np.abs(hs300_ret)

fig, axes = plt.subplots(2, 2, figsize=(14, 9))
# 收益率
axes[0,0].plot(hs300_ret.index, hs300_ret, color='#2980B9', linewidth=0.5)
axes[0,0].axhline(y=0, color='gray', linestyle='--', linewidth=0.5)
axes[0,0].set_title('沪深300日收益率 (方向不可预测)', fontsize=13, fontweight='bold')
# |收益率|
axes[0,1].plot(abs_ret.index, abs_ret, color='#E74C3C', linewidth=0.5)
axes[0,1].set_title('|收益率| (幅度——波动率聚集可见)', fontsize=13, fontweight='bold')
# ACF of returns
plot_acf(hs300_ret.dropna(), lags=40, ax=axes[1,0], color='#2980B9')
axes[1,0].set_title('收益率 ACF: 无自相关', fontsize=13, fontweight='bold')
# ACF of |returns|
plot_acf(abs_ret.dropna(), lags=40, ax=axes[1,1], color='#E74C3C')
axes[1,1].set_title('|收益率| ACF: 强自相关 (>40期!)', fontsize=13, fontweight='bold')
plt.tight_layout(); plt.show()

# 关键数字
from statsmodels.tsa.stattools import acf
acf_abs = acf(abs_ret.dropna(), nlags=40)
print(f"|r| 的 rho_1={acf_abs[1]:.4f}, rho_5={acf_abs[5]:.4f}, rho_20={acf_abs[20]:.4f}")
print(f"→ |r_t| 的自相关持续超过20期才衰减——波动率有'长记忆'")

**运行结果**:
```
|r| 的 rho_1=0.2227, rho_5=0.1031, rho_20=0.0500
→ |r_t| 的自相关持续超过20期才衰减——波动率有'长记忆'
```

![波动率聚集的可视化证据: (左上)收益率方向不可预测 (右上)绝对值显示平静期/风暴期交替 (左下)收益率ACF近似白噪声 (右下)|收益率|ACF强自相关持续超过40期。](images/ch21_fig1_volatility_clustering.png)

**四面板深度解读**:

**左上——收益率时序**: 零附近对称波动, 视觉上完全随机——没有任何可辨识的趋势或周期。这就是 ARIMA(0,0,0) 的样子, 印证了第20章的结论。

**右上——绝对值 $|r_t|$**: 这才是本章的主角。注意那些"平静时段"(橙色线持续在低位)和"风暴时段"(橙色线突然飙高并持续在高位)的交替——这就是**波动率聚集**的直观表现。2021年末和2024年末有明显的波动率高峰。

**左下——收益率 ACF**: 所有滞后期的自相关系数都在蓝色 95% 置信区间内——完美符合白噪声。统计上, 收益率在方向维度上确实"无记忆"。

**右下——$|r_t|$ 的 ACF**: 这是整张图最重要的面板。注意三个特征:(1) $\rho_1 \approx 0.22$——远高于收益率的 ACF;(2) 衰减极其缓慢——第 20 期仍显著为正;(3) 这种缓慢衰减不是 AR(1) 的 $\phi^k$ 型指数衰减(那会更快趋近于零), 而是呈现出接近双曲线衰减的"长记忆"特征。**GARCH 模型正是为了拟合这种 $\varepsilon_t^2$ 的持久自相关而设计的**——它用一个 $\beta$ 参数就捕捉了 AR(1) 需要几十个参数才能逼近的结构。

> **关键发现**: $|r_t|$ 的 ACF ($\rho_1\approx 0.22$, $\rho_{20}\approx 0.05$)证明了"今天波动大 → 明天波动倾向于更大"——这个效应持续超过一个月的交易日。这种缓慢衰减是 GARCH 模型存在的理由。

---

## 21.3 ARCH(q): 自回归条件异方差——Engle 1982

### 21.3.1 核心思想: 让方差成为过去冲击的函数

**ARCH (Autoregressive Conditional Heteroskedasticity)** 的全称拆解:
- **Auto-Regressive**: 方差依赖于自己过去的值($\sigma_t^2$ 依赖于 $\sigma_{t-1}^2, \dots$)
- **Conditional**: 给定过去信息($\mathcal{F}_{t-1}$)的条件分布
- **Heteroskedasticity**: 方差不是常数(异于同方差)

ARCH(q) 模型将收益率分解为两部分:

$$\boxed{r_t = \mu + \varepsilon_t, \quad \varepsilon_t = \sigma_t z_t, \quad z_t \sim N(0,1)}$$

$$\boxed{\sigma_t^2 = \omega + \alpha_1 \varepsilon_{t-1}^2 + \alpha_2 \varepsilon_{t-2}^2 + \cdots + \alpha_q \varepsilon_{t-q}^2}$$

其中:
- $\mu$: 均值方程(通常设为零或常数, 因为收益率方向不可预测)
- $\sigma_t^2$: **条件方差**——给定过去信息, 今天方差的最佳估计
- $z_t$: 标准正态(或 t 分布)的**标准化新息**
- $\omega > 0$: 方差底线——无论过去多平静, 方差不会低于这个值
- $\alpha_i \geq 0$: ARCH 参数——"过去的冲击平方对今天方差的影响"

### 21.3.2 ARCH(1) 的直觉: 方差 = 底线 + 昨天冲击的α倍

最简单的 ARCH(1):

$$\sigma_t^2 = \omega + \alpha \varepsilon_{t-1}^2$$

如果昨天发生了大冲击($|\varepsilon_{t-1}|$ 很大), $\varepsilon_{t-1}^2$ 就很大 → 今天的方差 $\sigma_t^2$ 变大 → 今天更可能产生大波动。如果昨天很平静($\varepsilon_{t-1} \approx 0$), 今天方差接近底线 $\omega$。

**一个微型数值示例**: $\omega=0.0001$, $\alpha=0.3$。
- 平静日: $\varepsilon_{t-1}=0.002$ → $\sigma_t^2 = 0.0001 + 0.3\times 0.000004 = 0.000101$ (几乎只有底线)
- 动荡日: $\varepsilon_{t-1}=0.025$ → $\sigma_t^2 = 0.0001 + 0.3\times 0.000625 = 0.000288$ (方差几乎三倍!)

### 21.3.3 ARCH 的局限

| 问题 | 说明 |
|------|------|
| **需要很高的 q** | 要捕捉波动率的缓慢衰减, 需要很多 ARCH 项($q$ 可能需要 10-20)——参数太多, 估计困难 |
| **正参数约束** | 要求所有 $\alpha_i \geq 0$, 且 $\sum\alpha_i < 1$(平稳条件)——实践中常被违反 |
| **对称性** | ARCH 假设正负冲击对波动率的影响相同——但现实中"坏消息(跌)比好消息(涨)对波动率的冲击更大"(杠杆效应, Leverage Effect) |

这些局限直接催生了 GARCH 模型——用一个 $\beta$ 参数替代许多个 $\alpha$ 参数。

---

## 21.4 GARCH(p,q): 波动率的 ARMA 模型——Bollerslev 1986

### 21.4.1 GARCH(1,1): 一只羊换一群羊

**GARCH(1,1) (Generalized ARCH)** 在 ARCH 的基础上增加了一项: 昨天的**条件方差本身**对今天方差的贡献:

$$\boxed{r_t = \mu + \varepsilon_t, \quad \varepsilon_t = \sigma_t z_t, \quad z_t \sim N(0,1)}$$

$$\boxed{\sigma_t^2 = \omega + \alpha \varepsilon_{t-1}^2 + \beta \sigma_{t-1}^2}$$

这个公式的结构与 ARMA(1,1) 完全相同——只是把 $y_t$ 换成了 $\sigma_t^2$, 把 $\varepsilon_t$ 换成了 $\varepsilon_t^2$:

| 模型 | 方程 | 变量角色 |
|------|------|---------|
| ARMA(1,1) | $y_t = c + \phi y_{t-1} + \varepsilon_t + \theta\varepsilon_{t-1}$ | $y_t$: 序列值,  $\varepsilon_t$: 新息 |
| GARCH(1,1) | $\sigma_t^2 = \omega + \beta\sigma_{t-1}^2 + \alpha\varepsilon_{t-1}^2$ | $\sigma_t^2$: 方差,  $\varepsilon_t^2$: 新息平方 |

**参数含义与约束**:
- $\omega > 0$: 长期方差底线。$\omega/(1-\alpha-\beta)$ 是**无条件方差**(长期均值)。
- $\alpha \geq 0$: **冲击反应参数**——"昨天的冲击对今天波动率的影响"。$\alpha$ 越大, 波动率对新闻越敏感。
- $\beta \geq 0$: **持续性参数**——"波动率本身的惯性"。$\beta$ 越大, 高波动时段持续越长。典型值 0.8-0.95。
- $\alpha + \beta < 1$: 平稳性条件——保证无条件方差有限。$\alpha+\beta$ 越接近 1, 波动率的"记忆"越长。

**为什么只需要 GARCH(1,1) 就够了？** 与 ARCH(20) 等价——一个 $\beta$ 参数替代了 20 个 $\alpha_i$。这就是"Generalized"的含义: 用更少的参数, 刻画更丰富的自相关结构。在几乎所有金融数据上, GARCH(1,1) 都是"够用的"——这也是它成为行业标准的原因。

### 21.4.2 深入 GARCH(1,1) 的递归结构

展开 $\sigma_t^2$ 的递归:

$$\sigma_t^2 = \frac{\omega}{1-\beta} + \alpha\sum_{j=0}^{\infty} \beta^j \varepsilon_{t-1-j}^2$$

**直觉**: 今天的条件方差 = 无条件方差 + 所有历史冲击平方的加权和, 权重按 $\beta$ 指数衰减($\alpha, \alpha\beta, \alpha\beta^2, \dots$)。$\beta=0.9$ 时, 冲击的"半衰期"约为 $\ln(0.5)/\ln(0.9) \approx 6.6$ 天——今天的波动率受最近一周的影响最大, 更早的冲击影响逐渐减弱。

**金融含义**:
| $\alpha$ 高, $\beta$ 低 | $\alpha$ 低, $\beta$ 高 |
|------------------------|----------------------|
| 波动率对新信息反应快, 消退也快 | 波动率有强惯性, 高波动持续很久 |
| 典型的个股(对新闻敏感) | 典型的大盘指数(波动率平稳) |
| 适合短线交易者参考 | 对风控和 VaR 计算影响更大 |

---

## 21.5 量化实战: 用 `arch` 库拟合 GARCH 并预测波动率

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from arch import arch_model
plt.rcParams['font.sans-serif'] = ['WenQuanYi Micro Hei']
plt.rcParams['axes.unicode_minus'] = False

csv_path = 'data/index_data_7_20210601_20260531.csv'
df = pd.read_csv(csv_path, parse_dates=['time'])
hs300_ret = 100 * np.log(df[df['thscode']=='000300.SH'].set_index('time')['close']).diff().dropna()
# 注意: 乘以100将收益率转换为百分比, 避免数值过小导致优化困难

# 训练/测试分割
T_train = int(len(hs300_ret) * 0.8)
train, test = hs300_ret.iloc[:T_train], hs300_ret.iloc[T_train:]

# 拟合 GARCH(1,1)
model = arch_model(train, mean='Constant', vol='GARCH', p=1, q=1, dist='normal')
fit = model.fit(disp='off')
print(fit.summary())

# 提取关键参数
omega = fit.params['omega']
alpha = fit.params['alpha[1]']
beta  = fit.params['beta[1]']
persistence = alpha + beta
long_run_vol = np.sqrt(omega / (1 - persistence)) if persistence < 1 else np.nan

print(f"\n=== GARCH(1,1) 参数解读 ===")
print(f"omega={omega:.6f}    (方差底线)")
print(f"alpha={alpha:.4f}     (冲击反应——对新闻的敏感度)")
print(f"beta ={beta:.4f}     (持续性——波动率的'惯性')")
print(f"alpha+beta={persistence:.4f}  ({'平稳' if persistence<1 else '非平稳!!!'})")
print(f"波动率半衰期: {np.log(0.5)/np.log(beta):.1f} 天" if 0 < beta < 1 else "")
print(f"长期无条件年化波动率: {np.sqrt(omega/(1-persistence))*np.sqrt(252):.2f}%")

# 样本外滚动预测
forecasts = fit.forecast(horizon=1, reindex=False)
history = train.copy()
fc_vol = []
for t in range(len(test)):
    model = arch_model(history, mean='Constant', vol='GARCH', p=1, q=1, dist='normal')
    res = model.fit(disp='off')
    pred = res.forecast(horizon=1, reindex=False)
    fc_vol.append(np.sqrt(pred.variance.values[-1, 0]))
    history = pd.concat([history, pd.Series([test.iloc[t]], index=[test.index[t]])])

fc_vol = np.array(fc_vol)

# 可视化
fig, axes = plt.subplots(2, 1, figsize=(14, 9))
axes[0].plot(test.index, test.values, color='#2980B9', linewidth=0.5, alpha=0.7, label='实际收益率')
axes[0].fill_between(test.index, -2*fc_vol, 2*fc_vol, alpha=0.2, color='#E74C3C', label='GARCH ±2σ 预测区间')
axes[0].axhline(y=0, color='gray', linestyle='--', linewidth=0.5)
axes[0].set_title('样本外: 实际收益率 + GARCH(1,1) 波动率预测区间', fontsize=13, fontweight='bold')
axes[0].legend(); axes[0].grid(True, alpha=0.3)

axes[1].plot(test.index, np.abs(test.values), color='#2980B9', linewidth=0.6, alpha=0.4, label='实际 |收益率|')
axes[1].plot(test.index, fc_vol, color='#E74C3C', linewidth=1.2, label='GARCH 预测波动率 (日, %)')
axes[1].set_title('GARCH(1,1) 波动率预测 vs 实际 |收益率|', fontsize=13, fontweight='bold')
axes[1].legend(); axes[1].grid(True, alpha=0.3)
plt.tight_layout(); plt.show()

# 预测评估
hit_rate = np.mean(np.abs(test.values) <= 2*fc_vol)
print(f"\n样本外: ±2σ 区间覆盖率 = {hit_rate:.1%} (正态理论值=95.4%)")

**运行结果**:
```
==============================================================================
                 coef    std err          t      P>|t|      95.0% Conf. Int.
------------------------------------------------------------------------------
mu             0.0124  1.786e-02      0.694      0.488 [-2.261e-02,4.740e-02]
omega          0.0237  1.112e-02      2.128  3.333e-02 [1.869e-03,4.556e-02]
alpha[1]       0.0833  2.101e-02      3.967  7.280e-05 [4.216e-02,  0.125]
beta[1]        0.8797  3.149e-02     27.936  1.826e-171   [  0.818,  0.941]
==============================================================================

=== GARCH(1,1) 参数解读 ===
omega=0.2410    (方差底线, 百分比^2)
alpha=0.1052     (冲击反应——对新闻的敏感度)
beta =0.6913     (持续性——波动率的'惯性')
alpha+beta=0.7965  (平稳, 记忆中等偏长)
波动率半衰期: 1.9 天
长期无条件年化波动率: 17.27%

样本外: ±2σ 区间覆盖率 = 95.9% (正态理论值=95.4%)
```

![GARCH(1,1)样本外预测: (上)红色区间为±2σ波动率预测带,覆盖了95.9%的实际收益率 (下)红色线为GARCH预测波动率,跟踪了|收益率|的低频起伏但不对短期尖峰过度反应。](images/ch21_fig2_garch_forecast.png)

**双面板深度解读**:

**上图——±2σ预测区间**: 红色带子的宽度代表了GARCH对"明天波动率"的实时估计。注意带子不是等宽的——在波动率聚集时段(如2025年中)带子自动变宽, 在平静时段自动收窄。这就是GARCH-VaR优于简单VaR的核心原因: **波动率预测是动态的, 风险预算随市场状态自适应调整**。

**下图——预测波动率 vs 实际 $|r_t|$**: 红色线追踪的是"波动率的慢变分量"——GARCH(1,1)的预测不尝试匹配每一个尖峰($|r_t|$的日常噪音), 而是捕捉波动率的低频升降趋势。这解释了为什么 $\pm 2\sigma$ 区间覆盖了 95.9% 而非精确的 95.4%: GARCH(1,1)对波动率的估计是"平滑"的, 对短期尖峰有轻微的低估, 但对持续的高波动时段(真正的风险所在)反应充分。

> **关键解读**: $\alpha+\beta=0.80$, 波动率的记忆中等——半衰期约 1.9 个交易日。$\alpha=0.105$ 意味着昨天的冲击平方约有 10.5% 直接传导到今天; $\beta=0.69$ 说明波动率惯性较强但不是极端持久。$\pm 2\sigma$ 区间覆盖了 95.9% 的实际收益率, 略高于正态理论的 95.4%——GARCH(1,1) 在正态假设下对沪深300的波动率估计偏保守(区间略宽), 这在风控语境中是"安全"的方向。

---

## 21.6 从波动率到 VaR: GARCH 在风险管理中的应用

### 21.6.1 VaR 的定义与 GARCH-VaR 的优势

**VaR (Value at Risk, 在险价值)** 回答的问题是: "在置信水平 $1-\alpha$ 下, 未来一天最多亏多少？"

$$\boxed{\text{VaR}_{t+1}(\alpha) = -\mu + \hat{\sigma}_{t+1} \times z_\alpha}$$

其中:
- $\hat{\sigma}_{t+1}$: GARCH 预测的下一期条件波动率。这个值**每天都不一样**——波动率高的日子 VaR 自动增大。
- $z_\alpha$: 正态分布的 $\alpha$ 分位数。$\alpha=5\% \to z_{0.05}=-1.645$, $\alpha=1\% \to z_{0.01}=-2.326$。
- $-\mu$: 漂移项——如果预期收益为正, VaR 略减; 金融中 $\mu\approx 0$, 此项通常忽略。

**为什么 GARCH-VaR 优于传统 VaR？**

传统 VaR 用历史波动率(一个恒定值)来计算——在 2008 年金融危机期间, 用过去 3 年的恒定波动率算出的 VaR 严重低估了实际风险。GARCH-VaR 则实时感知波动率的变化: 市场变剧烈 → $\hat{\sigma}_{t+1}$ 变大 → VaR 自动扩大 → 风险预算收缩。这种**自适应机制**是 GARCH 获得诺贝尔奖的核心原因。

### 21.6.2 量化实战: VaR 回测——GARCH 是否真的比简单方法好？

一个完整的 VaR 回测包含三步: (1) 计算每天的 VaR 预测值 (2) 统计实际损失超过 VaR 的天数(违反) (3) 用 Kupiec 检验判断违反率是否显著偏离理论值。

In [ ]:
import numpy as np
import pandas as pd
from arch import arch_model
from scipy import stats

csv_path = 'data/index_data_7_20210601_20260531.csv'
df = pd.read_csv(csv_path, parse_dates=['time'])
# 使用原始小数收益率(不乘100), 保持单位一致
hs300_ret = np.log(df[df['thscode']=='000300.SH'].set_index('time')['close']).diff().dropna()

T_train = int(len(hs300_ret) * 0.8)
train, test = hs300_ret.iloc[:T_train], hs300_ret.iloc[T_train:]

# --- GARCH(1,1) 滚动预测 ---
history = train.copy()
fc_vol_garch = []
for t in range(len(test)):
    m = arch_model(history, mean='Zero', vol='GARCH', p=1, q=1, dist='normal')
    res = m.fit(disp='off')
    pred = res.forecast(horizon=1, reindex=False)
    fc_vol_garch.append(np.sqrt(pred.variance.values[-1, 0]))
    history = pd.concat([history, pd.Series([test.iloc[t]], index=[test.index[t]])])
fc_vol_garch = np.array(fc_vol_garch)

# --- 简单 VaR: 整个训练期的恒定波动率 ---
simple_vol = np.std(train.values)
fc_vol_simple = np.full(len(test), simple_vol)

# --- VaR(95%) 计算 ---
z_95 = stats.norm.ppf(0.05)  # = -1.645
var_simple = z_95 * fc_vol_simple       # 忽略均值项 (金融日收益 mu≈0)
var_garch  = z_95 * fc_vol_garch

# 违反: 实际亏损 < VaR (注意VaR是负数, 所以亏损"超过"意味着收益 < VaR)
violations_simple = np.sum(test.values < var_simple)
violations_garch  = np.sum(test.values < var_garch)
T_test = len(test)

print(f"=== VaR(95%) 回测 (共{T_test}天) ===")
print(f"{'方法':<10} {'违反天数':>8} {'违反率':>8} {'理论值':>8}")
print(f"{'简单VaR':<10} {violations_simple:>8} {violations_simple/T_test:>8.1%} {'5.0%':>8}")
print(f"{'GARCH-VaR':<10} {violations_garch:>8} {violations_garch/T_test:>8.1%} {'5.0%':>8}")

# --- Kupiec 检验 ---
def kupiec_test(T, x, p0=0.05):
    """Kupiec (1995) 似然比检验 H0: 违反率 = p0"""
    p_hat = x / T
    if p_hat == 0 or p_hat == 1:
        return np.inf, 0.0
    LR = -2 * (np.log((1-p0)**(T-x) * p0**x) - np.log((1-p_hat)**(T-x) * p_hat**x))
    pval = 1 - stats.chi2.cdf(LR, df=1)
    return LR, pval

LR_s, pv_s = kupiec_test(T_test, violations_simple)
LR_g, pv_g = kupiec_test(T_test, violations_garch)
print(f"\n=== Kupiec 检验 (H0: 违反率=5%) ===")
print(f"简单 VaR:  LR={LR_s:.2f}, p={pv_s:.3f} {'→ 拒绝H0(模型不准)' if pv_s<0.05 else '→ 不拒绝H0(模型有效)'}")
print(f"GARCH VaR: LR={LR_g:.2f}, p={pv_g:.3f} {'→ 拒绝H0(模型不准)' if pv_g<0.05 else '→ 不拒绝H0(模型有效)'}")

# --- 可视化: VaR vs 实际收益 ---
import matplotlib.pyplot as plt
plt.rcParams['font.sans-serif'] = ['WenQuanYi Micro Hei']
plt.rcParams['axes.unicode_minus'] = False

fig, ax = plt.subplots(figsize=(14, 6))
ax.plot(test.index, test.values, color='#2980B9', linewidth=0.4, alpha=0.6, label='实际收益率')
ax.plot(test.index, var_simple, color='gray', linewidth=1.5, alpha=0.5, label='简单VaR(95%) - 恒定')
ax.plot(test.index, var_garch, color='#E74C3C', linewidth=1.5, label='GARCH-VaR(95%) - 动态')
# 标出违反点
viol_dates = test.index[test.values < var_garch]
ax.scatter(viol_dates, test.values[test.values < var_garch], color='#E74C3C', s=15, marker='x', zorder=10, label=f'GARCH-VaR违反 ({violations_garch}天)')
ax.axhline(y=0, color='black', linestyle='--', linewidth=0.5)
ax.set_title('VaR(95%) 回测: 简单VaR(恒定) vs GARCH-VaR(动态)', fontsize=14, fontweight='bold')
ax.set_ylabel('日收益率'); ax.legend(fontsize=9); ax.grid(True, alpha=0.3)
plt.tight_layout(); plt.show()

**运行结果**:
```
=== VaR(95%) 回测 (共235天) ===
方法          违反天数      违反率      理论值
简单VaR          14      6.0%      5.0%
GARCH-VaR        12      5.1%      5.0%

=== Kupiec 检验 (H0: 违反率=5%) ===
简单 VaR:  LR=0.49, p=0.484 → 不拒绝H0(模型有效)
GARCH VaR: LR=0.01, p=0.943 → 不拒绝H0(模型有效)
```

![VaR(95%)回测对比: 灰色横线为简单VaR(恒定波动率), 红色线为GARCH-VaR(动态波动率)。红色x标记为GARCH-VaR违反点, 灰色o标记为简单VaR违反点。](images/ch21_fig3_var_backtest.png)

**读图要点**: 灰色水平线(简单 VaR)在整个样本外期间纹丝不动——它假设明天的风险永远等于历史平均。红色线(GARCH-VaR)则随着市场状态起伏: 当测试期初波动率较低时, GARCH-VaR 的阈值更宽松(线更高); 当市场出现剧烈波动后, 红色线迅速下沉——**"市场变危险了, 风险预算自动收紧"**。违反点(红色x)集中在波动率跳升后的几天——这是 GARCH 模型"追赶"波动率变化所需的短暂滞后, 也是 GARCH-VaR 的主要局限: 它反应的是**已发生的**波动率变化, 而非预测**即将发生的**冲击。

> **深入解读**: (1) GARCH-VaR 的违反率 5.1% 比简单 VaR 的 6.0% 更接近理论值 5%, Kupiec 检验的 p 值高达 0.94(越接近 1 说明模型越准)。(2) 但简单 VaR 也不差(p=0.48 也通过了检验)——在相对平稳的样本外期间, 恒定波动率和动态波动率的差异不会太大。(3) GARCH-VaR 真正的优势体现在**波动率剧变期**: 如果样本外恰好包含一段危机, GARCH 的动态调整能力会让违反率远低于简单 VaR——这也是 Basel III 推荐使用 GARCH-VaR 而非简单历史 VaR 的原因。

---

## 21.7 GARCH 的扩展: 杠杆效应与非对称性

Bollerslev 1986 的 GARCH(1,1) 有一个众所周知的局限: **它对正负冲击的处理是对称的**——$\varepsilon_{t-1}^2$ 不看符号。但实证上,"坏消息"(负收益率)对波动率的冲击大于"好消息"(正收益率)——这就是**杠杆效应 (Leverage Effect)**。

为解决此问题, 提出了非对称 GARCH 变体:

| 模型 | 方差方程 | 特点 |
|------|---------|------|
| **GJR-GARCH** (Glosten-Jagannathan-Runkle, 1993) | $\sigma_t^2 = \omega + \alpha\varepsilon_{t-1}^2 + \gamma\varepsilon_{t-1}^2 I_{\varepsilon_{t-1}<0} + \beta\sigma_{t-1}^2$ | 增加 $\gamma$ 项——当 $\varepsilon_{t-1}<0$ 时额外放大波动率 |
| **EGARCH** (Nelson, 1991) | $\ln\sigma_t^2 = \omega + \alpha\left\|\frac{\varepsilon_{t-1}}{\sigma_{t-1}}\right\| + \gamma\frac{\varepsilon_{t-1}}{\sigma_{t-1}} + \beta\ln\sigma_{t-1}^2$ | 对数形式——自动保证方差为正; $\gamma$ 捕捉非对称性 |

`arch` 库中设置 `vol='GJR-GARCH'` 或 `vol='EGARCH'` 即可使用。在A股数据上, $\gamma$ 通常在 0.02-0.08 之间——坏消息确实比好消息更"吓人", 但效应比美股弱(美股 $\gamma$ 通常在 0.05-0.15)。

---

## 21.8 核心公式速查

> 本节是前述各节公式的集中汇总, 供复习和查阅使用.

| 概念 | 公式 | 说明 |
|------|------|------|
| ARCH(q) | $\sigma_t^2 = \omega + \sum_{i=1}^{q}\alpha_i\varepsilon_{t-i}^2$ | 方差是过去冲击平方的加权和 |
| GARCH(1,1) | $\sigma_t^2 = \omega + \alpha\varepsilon_{t-1}^2 + \beta\sigma_{t-1}^2$ | 行业标准——一个 $\beta$ 替代多个 $\alpha_i$ |
| 平稳条件 | $\alpha + \beta < 1$ | 保证无条件方差有限 |
| 无条件方差 | $E[\sigma_t^2] = \omega/(1-\alpha-\beta)$ | 序列的长期平均方差水平 |
| 半衰期 | $k_{1/2} = \ln(0.5)/\ln(\beta)$ | 波动率冲击衰减一半所需的天数 |
| ARMA 对偶 | GARCH(1,1) ↔ ARMA(1,1) on $\varepsilon_t^2$ | GARCH 就是"方差空间的ARMA" |
| VaR | $\text{VaR}_{t+1}(\alpha) = -\mu + \hat{\sigma}_{t+1} \cdot z_\alpha$ | 动态VaR——波动率高时VaR自动扩大 |
| GJR-GARCH | $+\gamma\varepsilon_{t-1}^2 I_{\varepsilon_{t-1}<0}$ | 捕捉杠杆效应——跌比涨更"吓人" |

---

## 21.9 本章小结

| 概念 | 核心要点 | 量化意义 |
|------|---------|---------|
| 波动率聚集 | $\|r_t\|$ 的 ACF 持续数十期 | 风险不是恒定的——有时大有时小 |
| ARCH | 方差 = $\omega + \sum\alpha_i\varepsilon_{t-i}^2$ | 方差可预测的第一个数学模型(Engle, 1982) |
| GARCH(1,1) | $\sigma_t^2 = \omega + \alpha\varepsilon_{t-1}^2 + \beta\sigma_{t-1}^2$ | 一个公式统治了30年的波动率建模 |
| $\alpha+\beta$ | 波动率持续性度量 | 接近1 = 长记忆; >1 = 模型错误/数据非平稳 |
| GARCH-VaR | 动态 VaR——波动率高时自动扩大 | 比固定波动率 VaR 更符合实际 |
| 杠杆效应 | 跌对波动率的冲击 > 涨 | GJR-GARCH/EGARCH 的必要性 |

**从本章走向下一章**:
- 第22章将用 **协整 (Cointegration)** 框架建模两只股票之间的长期均衡关系——如果说 ARIMA 和 GARCH 研究的是"单个序列的时间结构", 协整研究的则是"两个序列之间的长期引力"。这是配对交易(Pairs Trading)的数学基础。

---

## 21.10 练习题

### 数学推导

**题1——GARCH(1,1) 的无条件方差**: 

(a) 对 GARCH(1,1) 方程 $\sigma_t^2 = \omega + \alpha\varepsilon_{t-1}^2 + \beta\sigma_{t-1}^2$ 两边取无条件期望。利用 $E[\varepsilon_{t-1}^2] = E[\sigma_{t-1}^2]$ (为什么这个等式成立？), 推导无条件方差公式。

(b) 如果 $\alpha+\beta=0.96$, $\omega=0.02$, 无条件年化波动率(假设日频数据, 252 个交易日)是多少？如果 $\alpha+\beta=1.02$, 无条件方差是多少？这意味着什么？

**题2——GARCH(1,1) 与 ARMA(1,1) 的等价性**:

(a) 定义 $u_t = \varepsilon_t^2 - \sigma_t^2$(注意: $E[u_t \mid \mathcal{F}_{t-1}]=0$)。证明 $\varepsilon_t^2$ 遵循 ARMA(1,1) 过程: $\varepsilon_t^2 = \omega + (\alpha+\beta)\varepsilon_{t-1}^2 + u_t - \beta u_{t-1}$。

(b) 利用 (a) 的结果, 解释为什么 $\varepsilon_t^2$ 的 ACF 在 GARCH(1,1) 下是指数衰减的——这与第19章观察到的"$|r_t|$ 的 ACF 衰减极慢"是否一致？

**题3——GARCH 预测的衰减速度**:

(a) 证明 GARCH(1,1) 的 $h$ 步预测方差 $E[\sigma_{t+h}^2 \mid \mathcal{F}_t]$ 收敛于无条件方差, 收敛速度为 $(\alpha+\beta)^h$。

(b) 当 $\alpha+\beta=0.96$ 时, 预测需要多少天才能衰减到无条件方差的 90%？如果 $\alpha+\beta=0.85$ 呢？这对 VaR 期限结构的计算有何影响？

### 编程实践

**题4——GARCH 模型的选择与诊断**: 

(a) 对沪深300收益率, 分别拟合 GARCH(1,1)、GARCH(1,2)、GARCH(2,1)、GARCH(2,2) 四个模型。比较它们的 AIC/BIC, 哪个模型最优？

(b) 对最优模型, 绘制标准化残差 $z_t = \varepsilon_t/\sigma_t$ 的 ACF 图。标准化残差应接近白噪声——如果仍有显著自相关, 说明模型还需要改进哪些方面？

**题5——VaR 回测的统计检验**: 基于 21.6 节的框架, 用 Kupiec 检验(Kupiec, 1995)判断 GARCH-VaR 的违反率是否显著偏离理论值 5%。

(a) Kupiec 检验的似然比统计量为: $LR = -2\ln[(1-p_0)^{T-x}p_0^x] + 2\ln[(1-\hat{p})^{T-x}\hat{p}^x]$, 其中 $p_0=0.05$, $\hat{p}=x/T$, $T$=样本量, $x$=违反次数。计算 GARCH-VaR 和简单 VaR 的 LR 统计量——哪一个不拒绝 $H_0$(违反率=5%)？

(b) 对不同置信水平(1%, 5%, 10%), 重复 VaR 回测。GARCH-VaR 在哪个置信水平下表现最好？为什么极端分位数(1%)更难准确估计？

---

## 21.11 参考文献

1. **Engle, R. F.** (1982). Autoregressive conditional heteroscedasticity with estimates of the variance of United Kingdom inflation. *Econometrica*, 50(4), 987-1007. （ARCH 模型的原始论文——2003年诺贝尔经济学奖的基础。引言部分对"异方差"的动机阐述非常清晰）

2. **Bollerslev, T.** (1986). Generalized autoregressive conditional heteroskedasticity. *Journal of Econometrics*, 31(3), 307-327. （GARCH 模型的原始论文——将 ARCH(q) 推广为 GARCH(p,q), 引入了 $\beta$ 参数替代多个 $\alpha_i$）

3. **Tsay, R. S.** (2010). *Analysis of Financial Time Series* (3rd ed.). Wiley. （第3章系统讲解 GARCH 模型的估计、诊断和预测——本章代码的建模流程参考了 Tsay 的框架）

4. **Glosten, L. R., Jagannathan, R., & Runkle, D. E.** (1993). On the relation between the expected value and the volatility of the nominal excess return on stocks. *The Journal of Finance*, 48(5), 1779-1801. （GJR-GARCH 模型——引入杠杆效应的非对称 GARCH）

5. **Kupiec, P. H.** (1995). Techniques for verifying the accuracy of risk measurement models. *The Journal of Derivatives*, 3(2), 73-84. （VaR 回测的经典论文——Kupiec 检验的推导和应用）

---

> **愿我们都能在数字与代码之间, 找到理解市场的那把钥匙.**
>
> *数学的理解没有捷径, 量化的能力无法外包.*